# 05 — Skyline figures

Renders the two skyline figures used by the factsheet.


In [ ]:
# Load the figure dependencies and define the factsheet palette and output paths.
import json

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from eclipse_viewshed.project import repository_root
PROJECT_ROOT = repository_root()

from eclipse_viewshed import eclipse, solar

PROCESSED = PROJECT_ROOT / "data" / "processed"
PROFILES = PROCESSED / "horizon_profiles"
FIGDIR = PROJECT_ROOT / "reports" / "figures" / "factsheet"
FIGDIR.mkdir(parents=True, exist_ok=True)

# Define the factsheet palette and drawing units.
INK   = "#0E1117"
PANEL = "#1C232F"
RULE  = "#29323F"
TX    = "#E9ECF2"
TX2   = "#949DAF"
TX3   = "#7B8598"
SUN   = "#EEDD88"
WARM  = "#EE8866"

PT = 1 / 72
DPI = 300

print(f"profiles: {len(list(PROFILES.glob('*.csv')))} on disk")


## Inputs


In [ ]:
# Load the observer table, eclipse circumstances, and saved horizon profiles.
observers = pd.read_csv(PROJECT_ROOT / "data" / "external" / "observers.csv")


def profile_path(name: str) -> Path:
    """observers.csv name -> the filename notebook 04 wrote."""
    stem = "_".join(name.lower().replace(",", "").replace("-", " ").split())
    return PROFILES / f"{stem}.csv"


profiles = {}
for name in observers["name"]:
    path = profile_path(name)
    if path.exists():
        profiles[name] = pd.read_csv(path)
    else:
        print(f"  MISSING {path.name} - run notebook 04")

assert profiles, "no horizon profiles found; run notebook 04 first"

contacts = eclipse.contacts()
CEST = solar.UTC_OFFSET_HOURS
C1, C4 = contacts["first_contact_utc"], contacts["last_contact_utc"]
TMAX = contacts["maximum_utc"]

print(f"{len(profiles)} profiles loaded")
for name, p in profiles.items():
    print(f"  {name:<24} {len(p):>5} bearings   "
          f"horizon {p.horizon_deg.min():.2f} to {p.horizon_deg.max():.2f} deg")
print(f"\neclipse {solar.format_cest(C1 + CEST)} - {solar.format_cest(C4 + CEST)} CEST, "
      f"maximum {solar.format_cest(TMAX + CEST)} at "
      f"{100 * contacts['max_obscuration']:.1f}%")


## Drawing functions


In [ ]:
# Define the colour, eclipse-disc, connector, label, and frame drawing functions.
import matplotlib.colors as mcolors

def desat(colour, amount=0.45):
    """Pull a colour toward its own luminance. 0 leaves it, 1 makes it grey."""
    r, g, b = mcolors.to_rgb(colour)
    lum = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return tuple(c + (lum - c) * amount for c in (r, g, b))


def glow_below(ax, x, y, colour, depth=0.55, bands=14, peak=0.07, zorder=7):
    """A short wash of light cast downward from a skyline.

    Stacked translucent fills rather than a real gradient: matplotlib has no
    vertical alpha ramp for an arbitrary lower boundary, and 14 bands is
    already below where banding shows at 300 dpi.
    """
    step = depth / bands
    for k in range(bands):
        ax.fill_between(x, y - (k + 1) * step, y - k * step,
                        facecolor=colour, alpha=peak * (1 - k / bands),
                        linewidth=0, zorder=zorder)


# Wrap long margin labels onto two lines.
WRAP = 16


def label_right(ax, entries, min_gap, fontsize=8.0, centre_middle=False,
                top_raise=0.0, bottom_drop=0.0, label_offsets=None,
                global_offset=0.0):
    """Stack line names in the reserved margin without collisions.

    Mixed transform: x in axes fraction so the text starts just past the
    plotted area, y in data so the stack keeps the same physical spacing.
    `clip_on=False` is what lets it sit outside the axes at all.

    With `centre_middle`, the highest and lowest labels stay on their skyline
    endpoints. The middle labels form a compact group centred on their own
    endpoint heights, preserving the bunch visible in the plotted lines while
    maintaining a minimum baseline gap.
    """
    ordered = sorted(entries, key=lambda e: e[2], reverse=True)
    if centre_middle and len(ordered) >= 3:
        middle = ordered[1:-1]
        centre = sum(e[2] for e in middle) / len(middle)
        offsets = [(len(middle) - 1) / 2 - i for i in range(len(middle))]
        ys = ([ordered[0][2] + top_raise]
              + [centre + offset * min_gap for offset in offsets]
              + [ordered[-1][2] - bottom_drop])
    else:
        centre = sum(e[2] for e in ordered) / len(ordered)
        ys = [centre + ((len(ordered) - 1) / 2 - i) * min_gap
              for i in range(len(ordered))]

    ys = [y + global_offset for y in ys]
    if label_offsets:
        ys = [y + label_offsets.get(entry[0], 0.0)
              for entry, y in zip(ordered, ys)]

    import textwrap
    trans = ax.get_yaxis_transform()
    for (name, colour, _), y_lab in zip(ordered, ys):
        ax.annotate(textwrap.fill(name, WRAP), (1.012, y_lab), xycoords=trans,
                    ha="left", va="center", color=colour, fontsize=fontsize,
                    linespacing=1.1, fontfamily="monospace", zorder=11,
                    clip_on=False)


def draw_eclipsed_sun(ax, hour_utc, sky, scale=1.0, zorder=5):
    """Sun disc with the moon subtracted, at its true place in the sky.

    `sky` is the colour the moon is painted in, so it must match whatever sits
    behind the disc. `scale` exaggerates the radii only.
    """
    alt_, az_ = solar.sun_altaz(hour_utc)
    circ = eclipse.circumstances(hour_utc)
    dx, dy = eclipse.disc_offset(hour_utc)

    sun_radius = circ.sun_radius_deg * scale
    sun_patch = Circle((az_, alt_), sun_radius, facecolor=SUN,
                       edgecolor="none", zorder=zorder)
    ax.add_patch(sun_patch)
    moon_patch = Circle((az_ + dx * scale, alt_ + dy * scale),
                        circ.moon_radius_deg * scale,
                        facecolor=sky, edgecolor="none", zorder=zorder + 1)
    moon_patch.set_clip_path(sun_patch)
    ax.add_patch(moon_patch)
    return alt_, az_, circ.obscuration


# Confirm the Moon's vertical orientation before drawing the eclipse discs.
dx_max, dy_max = eclipse.disc_offset(TMAX)
print(f"moon offset at maximum: dx {dx_max:+.4f} deg, dy {dy_max:+.4f} deg")
assert dy_max < 0, "moon is not below the sun at maximum; check disc_offset"
print("crescent opens upward")


## Lead skyline


In [ ]:
# Configure and render the lead skyline for Wandelterras, Het Steen.
LEAD = "Wandelterras, Het Steen"
assert LEAD in profiles, f"Missing horizon profile for {LEAD}"

AZ_LO = solar.WEDGE_AZ_MIN
# Extend the display frame one degree beyond the measured wedge.
AZ_HI = 295.0

# Place compass labels at their true azimuths within the frame.
COMPASS = {"WNW": 292.5}

# Sample a shared one-second solar path for both figures.
t = np.arange(C1, C4, 1 / 3600)
alt = np.array([solar.sun_altaz(x)[0] for x in t])
az = np.array([solar.sun_altaz(x)[1] for x in t])

# Derive both frame heights from the shared angular limits.
FIG_W_PT = 400

# Reserve a label strip only on the comparison frame.
BOX_LABELLED = dict(left=0.0, right=0.795, top=1.0, bottom=0.07)
BOX_FULL = dict(left=0.0, right=1.0, top=1.0, bottom=0.07)


ALT_LO = -1.0
TRACK_HEADROOM = 0.95
ALT_HI = solar.sun_altaz(C1)[0] + TRACK_HEADROOM


def deg_per_pt(box):
    """Horizontal angular scale for one axes point."""
    ax_w = FIG_W_PT * (box["right"] - box["left"])
    return (AZ_HI - AZ_LO) / ax_w


def figure_height(box):
    """Canvas height required for the shared limits at a 1:1 data aspect."""
    ax_w = FIG_W_PT * (box["right"] - box["left"])
    ax_h = ax_w * (ALT_HI - ALT_LO) / (AZ_HI - AZ_LO)
    return ax_h / (box["top"] - box["bottom"])


DEG_PER_PT = deg_per_pt(BOX_LABELLED)
LABEL_GAP = 13.5 * DEG_PER_PT

ALT_MAX, AZ_MAX = solar.sun_altaz(TMAX)

# Define the accessible skyline series palette.
SERIES = {
    "Wandelterras, Het Steen": "#E69F00",
    "Scheldekaaien Zuid":     "#3DBE91",
    "Nieuw Zuid":             "#CC79A7",
    "Droogdokkenpark":        "#56B4E9",
    "MAS panoramic platform": "#B8C4D8",
}
LEAD_COLOUR = SERIES[LEAD]
TRACK = desat(SUN, 0.35)

# Find when the Sun's lower limb first meets the lead skyline.
lead_horizon = np.interp(az, profiles[LEAD].azimuth_deg,
                         profiles[LEAD].horizon_deg)
sun_radius = eclipse.circumstances(TMAX).sun_radius_deg
contact = (t >= TMAX) & (alt - sun_radius <= lead_horizon)
T_SKYLINE_CONTACT = t[np.flatnonzero(contact)[0]]


def sky_frame(box=None, contact_time=None):
    """The frame both figures share: dark ground, sun track, eclipsed discs."""
    box = BOX_LABELLED if box is None else box
    fig_h_pt = figure_height(box)
    fig, ax = plt.subplots(figsize=(FIG_W_PT * PT, fig_h_pt * PT), dpi=DPI)
    fig.patch.set_facecolor(INK)
    ax.set_facecolor(INK)

    inside = (alt < ALT_HI - 0.4) & (az >= AZ_LO) & (az <= AZ_HI)
    assert inside[0], "shared frame must include first contact"

    start, end = C1, t[inside][-1]
    disc_times = list(np.arange(start, end + 1e-9, 8 / 60))
    # Replace the nearest regular samples with the meaningful event times.
    for key_time in [TMAX] + ([contact_time] if contact_time is not None else []):
        nearest = int(np.argmin(np.abs(np.asarray(disc_times) - key_time)))
        disc_times[nearest] = key_time
    disc_times = sorted(set(disc_times))

    # Trim each connector to the adjacent solar-disc edges.
    centres = []
    for hour in disc_times:
        a, z = solar.sun_altaz(hour)
        r = eclipse.circumstances(hour).sun_radius_deg
        centres.append((z, a, r))
    for (z0, a0, r0), (z1, a1, r1) in zip(centres[:-1], centres[1:]):
        dz, da = z1 - z0, a1 - a0
        distance = np.hypot(dz, da)
        uz, ua = dz / distance, da / distance
        ax.plot([z0 + uz * r0, z1 - uz * r1],
                [a0 + ua * r0, a1 - ua * r1],
                color=TRACK, linewidth=0.8, alpha=0.65, zorder=3,
                solid_capstyle="butt")

    for hour in disc_times:
        draw_eclipsed_sun(ax, hour, sky=INK)

    label_times = [start, TMAX]
    if contact_time is not None:
        label_times.append(contact_time)
    for hour in label_times:
        a, z = solar.sun_altaz(hour)
        ax.annotate(solar.format_cest(hour + CEST), (z, a),
                    xytext=(6, 4), textcoords="offset points", color=TX2,
                    fontsize=8.2, ha="left", va="bottom", fontfamily="monospace",
                    zorder=10)

    ax.set_xlim(AZ_LO, AZ_HI)
    ax.set_ylim(ALT_LO, ALT_HI)
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_yticks([])
    ax.set_xticks([275, 280, 285, 290, 295])
    ax.set_xticklabels(["275°", "280°", "285°", "290°", "295°"], color=TX3,
                       fontsize=8.2, fontfamily="monospace")
    ax.get_xticklabels()[-1].set_ha("right")
    ax.tick_params(axis="x", length=0, pad=2)

    # Draw compass points along the upper edge.
    for name, azimuth in COMPASS.items():
        if not AZ_LO <= azimuth <= AZ_HI:
            continue
        ax.annotate(name, (azimuth, ALT_HI), xytext=(0, -4),
                    textcoords="offset points", color=TX2, fontsize=8.2,
                    ha="center", va="top", fontfamily="monospace",
                    zorder=10)
        ax.plot([azimuth, azimuth], [ALT_HI, ALT_HI - 0.22], color=TX3,
                linewidth=0.6, zorder=10)

    fig.subplots_adjust(**box)
    return fig, ax


def draw_skyline(ax, prof, colour, fill=False, fill_alpha=1.0,
                 linewidth=0.9, zorder=8):
    """One measured horizon: optional ground, glow, then the line.

    `fill_alpha` matters when more than one skyline is drawn. A solid fill
    hides every line that sits below its own crest, so on the comparison
    figure the ground is translucent and the lines behind it stay readable.
    """
    x = prof.azimuth_deg.to_numpy()
    y = prof.horizon_deg.to_numpy()
    if fill:
        ax.fill_between(x, ALT_LO, y, facecolor=PANEL, edgecolor="none",
                        alpha=fill_alpha, zorder=zorder - 2)
    glow_below(ax, x, y, colour, zorder=zorder - 1)
    ax.plot(x, y, color=colour, linewidth=linewidth, zorder=zorder,
            solid_joinstyle="round")
    return float(y[-1])


fig, ax = sky_frame(BOX_FULL, contact_time=T_SKYLINE_CONTACT)
# Draw the lead skyline without a redundant margin label.
draw_skyline(ax, profiles[LEAD], LEAD_COLOUR, fill=True, linewidth=1.0)

fig.savefig(FIGDIR / "fig1_skyline_lead.png", dpi=DPI, facecolor=INK)
print(f"frame: {AZ_HI - AZ_LO:.0f} deg wide x {ALT_HI - ALT_LO:.1f} deg tall, "
      f"label gap {LABEL_GAP:.2f} deg")
print(f"wrote {(FIGDIR / 'fig1_skyline_lead.png').relative_to(PROJECT_ROOT)}")
plt.show()


## Skyline comparison


In [ ]:
# Render the five skyline profiles with a shared solar path and aligned labels.
OTHERS = {
    name: colour for name, colour in SERIES.items() if name != LEAD
}
assert set(OTHERS) <= set(profiles), set(OTHERS) - set(profiles)

fig, ax = sky_frame(contact_time=T_SKYLINE_CONTACT)

entries = [(LEAD, LEAD_COLOUR,
            draw_skyline(ax, profiles[LEAD], LEAD_COLOUR, fill=True,
                         fill_alpha=0.45, linewidth=1.0))]
for name, colour in OTHERS.items():
    entries.append((name, colour,
                    draw_skyline(ax, profiles[name], colour, linewidth=0.85,
                                 zorder=9)))
label_right(ax, entries, LABEL_GAP, centre_middle=True,
            top_raise=9.0 * DEG_PER_PT,
            bottom_drop=14.0 * DEG_PER_PT,
            global_offset=-4.0 * DEG_PER_PT,
            label_offsets={
                "Droogdokkenpark": 4.0 * DEG_PER_PT,
                "MAS panoramic platform": 4.0 * DEG_PER_PT,
            })

fig.savefig(FIGDIR / "fig2_skylines_all.png", dpi=DPI, facecolor=INK)
print(f"wrote {(FIGDIR / 'fig2_skylines_all.png').relative_to(PROJECT_ROOT)}")
for name, _, y in entries:
    print(f"  {name:<24} ends at {y:.2f} deg")
plt.show()


## Visibility cross-check


In [ ]:
# Recalculate the published visibility table and compare it with notebook 04.
LIMB = 0.265
rows = []
for name, prof in profiles.items():
    hor_at_max = float(np.interp(AZ_MAX, prof.azimuth_deg, prof.horizon_deg))

    hor = np.interp(az, prof.azimuth_deg, prof.horizon_deg,
                    left=np.nan, right=np.nan)
    clear = np.isfinite(hor) & (alt + LIMB > hor)
    idx = np.flatnonzero(clear)
    end = idx[np.flatnonzero(np.diff(idx) > 1)[0]] if np.any(np.diff(idx) > 1) else idx[-1]

    rows.append({
        "name": name,
        "horizon_at_max_deg": round(hor_at_max, 2),
        "clearance_at_max_deg": round(ALT_MAX - hor_at_max, 2),
        "visible_at_max": bool(ALT_MAX > hor_at_max),
        "first_cest": solar.format_cest(t[idx[0]] + CEST),
        "last_cest": solar.format_cest(t[end] + CEST),
        "eclipse_minutes": round((t[end] - t[idx[0]]) * 60, 1),
        "clipped_by_wedge": bool(az[end] >= prof.azimuth_deg.max() - 0.05),
    })

# Sort by the whole minutes shown in the factsheet, then break ties by clearance.
table = pd.DataFrame(rows)
table["_display_minutes"] = table["eclipse_minutes"].round()
table = (table
         .sort_values(["_display_minutes", "clearance_at_max_deg"],
                      ascending=[False, False])
         .drop(columns="_display_minutes"))
out = PROCESSED / "05_factsheet_table.csv"
table.to_csv(out, index=False)
print(f"wrote {out.relative_to(PROJECT_ROOT)}\n")
print(table.to_string(index=False))

print(f"\nwhole eclipse, unobstructed: {(C4 - C1) * 60:.0f} min "
      f"({solar.format_cest(C1 + CEST)} to {solar.format_cest(C4 + CEST)})")

# Verify that the independent notebook 04 calculation agrees.
reference = pd.read_csv(PROCESSED / "04_visibility.csv").set_index("name")
for _, row in table.iterrows():
    assert abs(
        reference.loc[row["name"], "clearance_at_max"]
        - row["clearance_at_max_deg"]
    ) <= 0.02
    assert abs(
        reference.loc[row["name"], "eclipse_minutes"]
        - row["eclipse_minutes"]
    ) <= 0.3
print("Notebook 04 and 05 visibility results agree.")
